In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import google.generativeai as genai
import os
import json
import re
import hashlib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from underthesea import word_tokenize
from tqdm import tqdm
import faiss
import pickle


c:\Users\tientm1\DS300-UIT-RecommenderSystem\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\tientm1\AppData\Local\Temp\ipykernel_11040\2435006755.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 1. Load data

In [2]:
# SETUP & DATA LOADING
csv_file = '../../data/all_recipes_final.csv'
data = pd.read_csv(csv_file)
data['doc_id'] = data.index.astype(int)

for column_name in ['title', 'description', 'ingredients_normalized']:
    data[column_name] = data[column_name].fillna('')

def compute_row_order_hash(title_series):
    """Create a stable fingerprint for the current recipe row order."""
    joined_titles = '\n'.join(title_series.astype(str).tolist())
    return hashlib.sha256(joined_titles.encode('utf-8')).hexdigest()

ROW_ORDER_TITLE_HASH = compute_row_order_hash(data['title'])
print(f"Loaded {len(data)} recipes.")
print(f"Row-order title hash: {ROW_ORDER_TITLE_HASH}")


Loaded 10263 recipes.
Row-order title hash: 6d7dd40daa64ffa15f4c7a51eb7cd19529768c373bcc68850961494bad0579e5


In [3]:
# Create the SBERT document text representation.
# This model intentionally uses title + normalized ingredients, because the method focuses on
# dish identity and ingredient-level semantic similarity.
data['sbert_document_text'] = (
    data['title'].astype(str).str.strip() + '. ' +
    data['ingredients_normalized'].astype(str).str.strip()
).str.strip()

# Keep the old column name for compatibility with later cells in this notebook.
data['combined_text'] = data['sbert_document_text']

print(f"Loaded {len(data)} recipes.")
data[['doc_id', 'title', 'ingredients_normalized', 'sbert_document_text']].head()


Loaded 10263 recipes.


,doc_id,title,ingredients_normalized,sbert_document_text
0,0,Cách muối dưa hành truyền thống,"{'tro bếp hoặc nước vo gọa', 'đường', 'cà rốt ...",Cách muối dưa hành truyền thống. {'tro bếp hoặ...
1,1,Su hào xào mực - món cổ Tết Bát Tràng,"{'muối', 'mỡ lợn hoặc dầu ăn', 'đường', 'gia v...",Su hào xào mực - món cổ Tết Bát Tràng. {'muối'...
2,2,Canh măng ngày Tết cổ truyền Hà Nội,"{'muối', 'móng giò lợn', 'nước vo gạo ngâm măn...","Canh măng ngày Tết cổ truyền Hà Nội. {'muối', ..."
3,3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,"{'bộ lòng mề gà', 'muối', 'mỡ lợn', 'gia vị: m...",Giả hạnh nhân - món ngon Tết xưa Hà Nội. {'bộ ...
4,4,Chả bì ớt xiêm xanh,"{'muối', 'gừng để sơ chế bì', 'bì lợn', 'hành ...","Chả bì ớt xiêm xanh. {'muối', 'gừng để sơ chế ..."


In [4]:
data.head()

,title,type_of_food,link,description,ingredients,ingredients_normalized,step,note,num_of_ingredients,cook_time,num_of_people,calories,source,doc_id,sbert_document_text,combined_text
0,Cách muối dưa hành truyền thống,Món Tết,https://vnexpress.net/doi-song-cooking-cach-mu...,Dưa hành muối là món ăn truyền thống ngày Tết ...,"['1 kg hành củ tươi', 'Tro bếp hoặc nước vo gọ...","{'tro bếp hoặc nước vo gọa', 'đường', 'cà rốt ...",['Bước 1: Chọn hành củ: Nên chọn hành củ ta bá...,[],5,45 phút,8-10 người,459 kcal,vnexpress,0,Cách muối dưa hành truyền thống. {'tro bếp hoặ...,Cách muối dưa hành truyền thống. {'tro bếp hoặ...
1,Su hào xào mực - món cổ Tết Bát Tràng,Món Tết,https://vnexpress.net/doi-song-cooking-su-hao-...,Đĩa xào khô ráo với su hào giòn ngọt quyện với...,"['2 củ su hào non', '1 con mực khô', '1/2 củ c...","{'muối', 'mỡ lợn hoặc dầu ăn', 'đường', 'gia v...",['Bước 1: Chọn và sơ chế mực: Người dân làng g...,['Su hào xào mực cùng với canh măng mực là hai...,6,50 phút,4 - 5 người,1.162 kcal,vnexpress,1,Su hào xào mực - món cổ Tết Bát Tràng. {'muối'...,Su hào xào mực - món cổ Tết Bát Tràng. {'muối'...
2,Canh măng ngày Tết cổ truyền Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-canh-ma...,"Măng ngấu vị, giòn ngon, móng giò hầm vừa độ s...","['800 gr măng khô', '2 móng giò lợn', 'Nước dù...","{'muối', 'móng giò lợn', 'nước vo gạo ngâm măn...","['Bước 1: Chọn măng khô: Theo lối cũ, người nộ...",['Nếu tận dụng nước luộc gà nấu canh măng thì ...,6,100 phút,8 - 10 người,4.930 kcal,vnexpress,2,"Canh măng ngày Tết cổ truyền Hà Nội. {'muối', ...","Canh măng ngày Tết cổ truyền Hà Nội. {'muối', ..."
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-gia-han...,Đây là món ăn cổ truyền thường thấy trong cỗ T...,"['2 bộ lòng mề gà', '100 gr lạc', '50 gr hạt đ...","{'bộ lòng mề gà', 'muối', 'mỡ lợn', 'gia vị: m...",['Bước 1: Chọn và sơ chế lạc: Chọn lạc khô chắ...,['Hạnh nhân xào (hay giả hạnh nhân) là món ăn ...,8,60 phút,4-5 người,1.112 kcal,vnexpress,3,Giả hạnh nhân - món ngon Tết xưa Hà Nội. {'bộ ...,Giả hạnh nhân - món ngon Tết xưa Hà Nội. {'bộ ...
4,Chả bì ớt xiêm xanh,Món Tết,https://vnexpress.net/doi-song-cooking-cha-bi-...,"Chả bì bóng đẹp, gói đều tay. Khi ăn vị ngọt m...","['500 gr giò sống', '300 gr bì lợn', '20 - 30 ...","{'muối', 'gừng để sơ chế bì', 'bì lợn', 'hành ...","['Bước 1: Chọn và sơ chế bì lợn, chuẩn bị giò ...",['Nên sơ chế kỹ bì lợn để chả được thơm. Tùy t...,6,60 phút,5-6 người,2.512 kcal,vnexpress,4,"Chả bì ớt xiêm xanh. {'muối', 'gừng để sơ chế ...","Chả bì ớt xiêm xanh. {'muối', 'gừng để sơ chế ..."


## 2. Method 1: SBERT + FAISS (Bi-Encoder Retrieval)

**Model-based approach using deep learning embeddings**
- Sử dụng Vietnamese SBERT để tạo semantic embeddings
- FAISS index cho fast similarity search
- Scalable và efficient cho large-scale retrieval

### 2.1. Initialize SBERT Model

In [5]:
# Load Vietnamese SBERT model
# Options: 'keepitreal/vietnamese-sbert', 'VoVanPhuc/sup-SimCSE-VietNamese-phobert-base', 
#          'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'

sbert_model_name = 'keepitreal/vietnamese-sbert'
print(f"Loading SBERT model: {sbert_model_name}")

sbert_model = SentenceTransformer(sbert_model_name)
print(f"Model loaded successfully! Embedding dimension: {sbert_model.get_sentence_embedding_dimension()}")

Loading SBERT model: keepitreal/vietnamese-sbert


c:\Users\tientm1\DS300-UIT-RecommenderSystem\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tientm1\.cache\huggingface\hub\models--keepitreal--vietnamese-sbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11698.69it/s]


Model loaded successfully! Embedding dimension: 768


C:\Users\tientm1\AppData\Local\Temp\ipykernel_11040\1152775006.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully! Embedding dimension: {sbert_model.get_sentence_embedding_dimension()}")


### 2.2. Encode All Recipes to Embeddings

In [6]:
# Encode all recipes (offline, only once)
print(f"Encoding {len(data)} recipes to embeddings...")

recipe_embeddings_sbert = sbert_model.encode(
    data['combined_text'].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True  # L2 normalize for cosine similarity
)

print(f"Embeddings shape: {recipe_embeddings_sbert.shape}")
print(f"Embedding dimension: {recipe_embeddings_sbert.shape[1]}")

Encoding 10263 recipes to embeddings...


Batches: 100%|██████████| 321/321 [22:56<00:00,  4.29s/it]

Embeddings shape: (10263, 768)
Embedding dimension: 768


### 2.3. Build FAISS Index

In [7]:
# Build FAISS index for fast similarity search
embedding_dim = recipe_embeddings_sbert.shape[1]

# Use IndexFlatIP for Inner Product (cosine similarity on normalized vectors)
faiss_index_sbert = faiss.IndexFlatIP(embedding_dim)

# Add all recipe embeddings to index
faiss_index_sbert.add(recipe_embeddings_sbert)

print(f"FAISS index built successfully!")
print(f"Total vectors in index: {faiss_index_sbert.ntotal}")

FAISS index built successfully!
Total vectors in index: 10263


### 2.4. Save SBERT Model, Embeddings, and FAISS Index

In [8]:
# Create directory for SBERT method
sbert_save_dir = '../Saved_models/SBERT_FAISS'
os.makedirs(sbert_save_dir, exist_ok=True)

# Save embeddings
embeddings_path = os.path.join(sbert_save_dir, 'recipe_embeddings.npy')
np.save(embeddings_path, recipe_embeddings_sbert)

# Save FAISS index
faiss_path = os.path.join(sbert_save_dir, 'faiss_index.bin')
faiss.write_index(faiss_index_sbert, faiss_path)

# Save model metadata for later loading and compatibility checks.
model_info = {
    'method': 'SBERT_FAISS',
    'model_name': sbert_model_name,
    'embedding_dim': int(embedding_dim),
    'num_recipes': int(len(data)),
    'normalize_embeddings': True,
    'dataset_path': csv_file,
    'row_order_title_hash': ROW_ORDER_TITLE_HASH,
    'text_representation': 'title + ingredients_normalized',
    'text_column': 'sbert_document_text'
}
info_path = os.path.join(sbert_save_dir, 'model_info.json')
with open(info_path, 'w', encoding='utf-8') as f:
    json.dump(model_info, f, indent=2, ensure_ascii=False)

print(f"\nSaved successfully: {sbert_save_dir}")



Saved successfully: ../Saved_models/SBERT_FAISS


### 2.5. Test SBERT + FAISS with Sample Queries

In [9]:
def sbert_search(query, k=10, threshold=0.5):
    """Search recipes using SBERT + FAISS"""
    # Encode query
    query_embedding = sbert_model.encode(
        [query], 
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    
    # Search in FAISS index
    scores, indices = faiss_index_sbert.search(query_embedding, k)
    
    # Filter by threshold
    valid_mask = scores[0] >= threshold
    valid_indices = indices[0][valid_mask]
    valid_scores = scores[0][valid_mask]
    
    return valid_indices, valid_scores

In [10]:
# Test queries
test_queries = [
    "Thịt kho nước dừa",
    "Canh chua cá lóc",
    "Món chay thanh đạm",
    "Bánh ngọt cho bữa sáng"
]

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-"*80)
    
    indices, scores = sbert_search(query, k=5, threshold=0.3)
    
    if len(indices) == 0:
        print("   No results found above threshold")
    else:
        for i, (idx, score) in enumerate(zip(indices, scores), 1):
            recipe = data.iloc[idx]
            print(f"   {i}. [{score:.3f}] {recipe['title']}")
            print(f"      {recipe['description'][:100]}...")
    print()


Query: 'Thịt kho nước dừa'
--------------------------------------------------------------------------------
   1. [0.756] Thịt heo kho cùi dừa
      Món thịt kho tàu hay thịt kho hột vịt quá quen thuộc với những bữa cơm gia đình rồi, nhưng đến hôm n...
   2. [0.715] Món cá lóc kho tộ cực dễ, thơm ngon đậm đà đưa cơm
      Trời bắt đầu se lạnh, bữa ăn gia đình với chén cơm nóng hổi, miếng cá kho thơm nồng thì không còn gì...
   3. [0.700] Thịt kho tàu miền Bắc ngon ăn Tết bằng nồi đơn
      Thịt kho tàu miền Bắc thơm ngon và có nét đặc trưng riêng biệt không lẫn vào đâu được. Hãy cùng Vào ...
   4. [0.688] Thịt kho dừa thơm ngon, ngọt béo, đậm đà đưa cơm
      Thịt kho dừa là món ăn vô cùng quen thuộc trong bữa cơm của nhiều gia đình vì vị đậm đà của thịt, ng...
   5. [0.687] Thịt kho Tàu miền Nam ngon ngọt thơm lừng, mềm ngon đậm vị
      Thịt kho Tàu là một trong những món kho từ lâu đã gắn liền với người Việt. Đối với mỗi vùng miền khá...


Query: 'Canh chua cá lóc'
----------------

## 3. Method 2: Hybrid TF-IDF + SBERT (Ensemble)

**Ensemble approach combining lexical and semantic matching**
- TF-IDF: Captures exact keyword matching
- SBERT: Captures semantic similarity
- Weighted combination: α × TF-IDF + (1-α) × SBERT

### 3.1. Load Pre-trained TF-IDF Model

In [11]:
# Load pre-trained TF-IDF artifacts from the content-based notebook.
tfidf_dir = '../Saved_models/TFIDF'

tfidf_vectorizer_path = os.path.join(tfidf_dir, 'tfidf_vectorizer.pkl')
tfidf_matrix_path = os.path.join(tfidf_dir, 'tfidf_matrix.pkl')
tfidf_data_path = os.path.join(tfidf_dir, 'tfidf_processed_data.pkl')
tfidf_metadata_path = os.path.join(tfidf_dir, 'metadata.json')

required_tfidf_files = [
    tfidf_vectorizer_path,
    tfidf_matrix_path,
    tfidf_data_path,
    tfidf_metadata_path
]
missing_tfidf_files = [path for path in required_tfidf_files if not os.path.exists(path)]
if missing_tfidf_files:
    raise FileNotFoundError(
        "Missing TF-IDF artifacts. Re-run 2_Content_based_methods.ipynb first. "
        f"Missing files: {missing_tfidf_files}"
    )

with open(tfidf_vectorizer_path, 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
with open(tfidf_matrix_path, 'rb') as f:
    tfidf_matrix = pickle.load(f)
with open(tfidf_data_path, 'rb') as f:
    tfidf_processed_data = pickle.load(f)
with open(tfidf_metadata_path, 'r', encoding='utf-8') as f:
    tfidf_metadata = json.load(f)

print("Loaded TF-IDF artifacts")
print(f"  - Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"  - Matrix shape: {tfidf_matrix.shape}")


Loaded TF-IDF artifacts
  - Vocabulary size: 5000
  - Matrix shape: (10263, 5000)


In [12]:
# Validate TF-IDF artifacts against the current recipe table before building the hybrid method.
if tfidf_matrix.shape[0] != len(data):
    raise ValueError(
        f"TF-IDF matrix row count ({tfidf_matrix.shape[0]}) does not match current data ({len(data)})."
    )

if tfidf_metadata.get('num_recipes') != len(data):
    raise ValueError(
        f"TF-IDF metadata num_recipes ({tfidf_metadata.get('num_recipes')}) does not match current data ({len(data)})."
    )

if tfidf_metadata.get('row_order_title_hash') != ROW_ORDER_TITLE_HASH:
    raise ValueError(
        "TF-IDF row order hash does not match the current dataset. "
        "Use the same all_recipes_final.csv and row order for both notebooks."
    )

if 'title' in tfidf_processed_data.columns:
    titles_match = tfidf_processed_data['title'].astype(str).reset_index(drop=True).equals(
        data['title'].astype(str).reset_index(drop=True)
    )
    if not titles_match:
        raise ValueError("TF-IDF processed data title order does not match the current dataset.")

print("TF-IDF artifact validation passed.")
print(f"  - Matrix sparsity: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")
print(f"  - Text representation: {tfidf_metadata.get('text_representation')}")


TF-IDF artifact validation passed.
  - Matrix sparsity: 92.52%
  - Text representation: title_clean + description_clean + step_clean + ingredients_clean


### 3.2. Save Hybrid Model Components

In [13]:
# Create directory for Hybrid method
hybrid_save_dir = '../Saved_models/Hybrid_TFIDF_SBERT'
os.makedirs(hybrid_save_dir, exist_ok=True)

# Save SBERT embeddings (new component)
hybrid_embeddings_path = os.path.join(hybrid_save_dir, 'sbert_embeddings.npy')
np.save(hybrid_embeddings_path, recipe_embeddings_sbert)
print(f"Saved SBERT embeddings to {hybrid_embeddings_path}")

Saved SBERT embeddings to ../Saved_models/Hybrid_TFIDF_SBERT\sbert_embeddings.npy


In [14]:
# Save model configuration (needed for evaluation phase)
hybrid_config = {
    'sbert_model_name': sbert_model_name,
    'tfidf_source': '../Saved_models/TFIDF',
    'alpha': 0.5,  # Fixed method definition. Do not tune this on the final test/qrels.
    'num_recipes': int(len(data)),
    'dataset_path': csv_file,
    'row_order_title_hash': ROW_ORDER_TITLE_HASH,
    'method': 'Hybrid_TFIDF_SBERT',
    'tfidf_text_representation': tfidf_metadata.get('text_representation'),
    'sbert_text_representation': 'title + ingredients_normalized',
    'description': 'Ensemble of TF-IDF lexical scores and SBERT semantic scores with a fixed weighted combination'
}
config_path = os.path.join(hybrid_save_dir, 'config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(hybrid_config, f, indent=2, ensure_ascii=False)

print(f"\nHybrid TF-IDF + SBERT method saved")
print(f"   Location: {hybrid_save_dir}")
print(f"   Components:")
print(f"      - SBERT embeddings (new)")
print(f"      - TF-IDF matrix and vectorizer reused from {tfidf_dir}")
print(f"      - Config with alpha={hybrid_config['alpha']}")



Hybrid TF-IDF + SBERT method saved
   Location: ../Saved_models/Hybrid_TFIDF_SBERT
   Components:
      - SBERT embeddings (new)
      - TF-IDF matrix and vectorizer reused from ../Saved_models/TFIDF
      - Config with alpha=0.5


### 3.3. Test Hybrid Method with Sample Queries

In [15]:
def hybrid_search(query, k=10, alpha=0.5, threshold=0.3):
    """
    Hybrid search combining TF-IDF and SBERT
    
    Args:
        query: Search query
        k: Number of top results
        alpha: Weight for TF-IDF (1-alpha for SBERT)
        threshold: Minimum combined score threshold
    
    Returns:
        indices: Recipe indices
        scores: Combined scores
        tfidf_scores: Individual TF-IDF scores
        sbert_scores: Individual SBERT scores
    """
    # 1. TF-IDF scoring
    query_tfidf = tfidf_vectorizer.transform([query])
    tfidf_scores = linear_kernel(query_tfidf, tfidf_matrix).flatten()
    
    # 2. SBERT scoring
    query_embedding = sbert_model.encode(
        [query], 
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    sbert_scores = cosine_similarity(query_embedding, recipe_embeddings_sbert).flatten()
    
    # 3. Combine scores
    combined_scores = alpha * tfidf_scores + (1 - alpha) * sbert_scores
    
    # 4. Get top-k
    top_indices = np.argsort(combined_scores)[::-1][:k]
    
    # 5. Filter by threshold
    valid_mask = combined_scores[top_indices] >= threshold
    valid_indices = top_indices[valid_mask]
    
    return (valid_indices, 
            combined_scores[valid_indices],
            tfidf_scores[valid_indices],
            sbert_scores[valid_indices])

In [16]:
alpha = 0.5  # Equal weight for TF-IDF and SBERT

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-"*80)
    
    indices, combined_scores, tfidf_scores, sbert_scores = hybrid_search(
        query, k=5, alpha=alpha, threshold=0.2
    )
    
    if len(indices) == 0:
        print("   No results found above threshold")
    else:
        for i, (idx, comb_score, tfidf_score, sbert_score) in enumerate(
            zip(indices, combined_scores, tfidf_scores, sbert_scores), 1
        ):
            recipe = data.iloc[idx]
            print(f"   {i}. [Combined: {comb_score:.3f}] (TF-IDF: {tfidf_score:.3f}, SBERT: {sbert_score:.3f})")
            print(f"      {recipe['title']}")
            print(f"      {recipe['description'][:100]}...")
    print()


Query: 'Thịt kho nước dừa'
--------------------------------------------------------------------------------
   1. [Combined: 0.628] (TF-IDF: 0.500, SBERT: 0.756)
      Thịt heo kho cùi dừa
      Món thịt kho tàu hay thịt kho hột vịt quá quen thuộc với những bữa cơm gia đình rồi, nhưng đến hôm n...
   2. [Combined: 0.603] (TF-IDF: 0.519, SBERT: 0.687)
      Thịt kho Tàu miền Nam ngon ngọt thơm lừng, mềm ngon đậm vị
      Thịt kho Tàu là một trong những món kho từ lâu đã gắn liền với người Việt. Đối với mỗi vùng miền khá...
   3. [Combined: 0.557] (TF-IDF: 0.426, SBERT: 0.688)
      Thịt kho dừa thơm ngon, ngọt béo, đậm đà đưa cơm
      Thịt kho dừa là món ăn vô cùng quen thuộc trong bữa cơm của nhiều gia đình vì vị đậm đà của thịt, ng...
   4. [Combined: 0.536] (TF-IDF: 0.372, SBERT: 0.700)
      Thịt kho tàu miền Bắc ngon ăn Tết bằng nồi đơn
      Thịt kho tàu miền Bắc thơm ngon và có nét đặc trưng riêng biệt không lẫn vào đâu được. Hãy cùng Vào ...
   5. [Combined: 0.533] (TF-IDF: 0.